# The end-to-end project

**Lecture 2** · Géron, Chapter 2

Applications of Machine Learning — BSc Mathematics of Artificial Intelligence

---

**How to use this notebook.** You are not expected to type the code. You are
expected to *read* it before you run it, and to be able to say what every line
does and what would break if it changed.

Every code cell is preceded by the **specification that would produce it** —
input, output, constraint, check. Read the box, work out what the check should
say, *then* run the cell. That order is the whole point of the box.

Run the cells in order. Anything that takes more than a few seconds says so,
and anything that needs a GPU says that too. Nothing here is wrong on purpose.

**About the prompt boxes.** Every code cell in this notebook is preceded by a
quoted prompt naming four things: the input, the output, the constraint the
method must respect, and a check whose answer you can work out before running
anything. Read the box, answer the check in your head, then run the cell.

The prompts are **specifications, not transcripts** — this is what you would
have to ask for in order to get this cell, not a recording of somebody asking
for it. If your own prompt is vaguer than the box, expect worse code than the
cell below it.


The whole pipeline, end to end: the preprocessing that has to be *learned* from
data, the cross-validation that gives an honest estimate of error, the search
that tunes it, and the single use of the test set at the end.

The first two cells repeat the Lecture 1 load and split, so this notebook stands
on its own. Runs on free CPU; the search cell takes two to four minutes.

## 1 · Setup, and the same split as last time

> **Prompt · setup**
>
> **input** · nothing
>
> **output** · the version of every library this notebook depends on, and one seed
>
> **constraint** · ASSERT the scikit-learn version rather than printing it — `root_mean_squared_error` arrived in 1.4, and on an older Colab image the failure is an ImportError twenty cells from here
>
> **check** · RANDOM_STATE is defined here ONCE and used for every split, every model and every shuffle in the notebook. A notebook carrying three different seeds cannot be reproduced by reading it

In [ ]:
# --- setup -------------------------------------------------------------------
# Not examinable: this is engineering hygiene, not machine learning. It is here
# because a version mismatch produces a confusing error twenty cells later.
import sys, sklearn, numpy as np, pandas as pd, matplotlib

print(f"python       {sys.version.split()[0]}")
print(f"scikit-learn {sklearn.__version__}")
print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")

# root_mean_squared_error arrived in scikit-learn 1.4
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), \
    "This notebook needs scikit-learn >= 1.4.  In Colab: %pip install -U scikit-learn"

RANDOM_STATE = 42          # every split, every model, every shuffle
pd.set_option("display.width", 100)

> **Prompt · the data**
>
> **input** · the California housing tarball
>
> **output** · 20,640 districts and 10 columns
>
> **constraint** · a FUNCTION that downloads if absent and reads if present — the data will change, and you will need this on another machine
>
> **check** · assert the shape, rather than trusting the download
>
> ---
>
> **try** · delete the `datasets/` directory and re-run the cell. If it cannot rebuild its own input from nothing, it is not reproducible — it is cached.

In [ ]:
# --- the data ----------------------------------------------------------------
# A function, not a manual download: the data will change, and you will need
# this on another machine.  ~5 s the first time, instant afterwards.
from pathlib import Path
import tarfile, urllib.request

def load_housing():
    tarball = Path("datasets/housing.tgz")
    if not tarball.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball)
        with tarfile.open(tarball) as t:
            t.extractall(path="datasets", filter="data")
    return pd.read_csv("datasets/housing/housing.csv")

housing_full = load_housing()

assert housing_full.shape == (20640, 10), f"unexpected shape {housing_full.shape}"
print(f"{len(housing_full):,} districts, {housing_full.shape[1]} columns")
housing_full.head()

> **Prompt · every import, and the same split**
>
> **input** · the same data and the same seed
>
> **output** · the identical 16,512 / 4,128 split as the previous lecture
>
> **constraint** · every import this notebook needs in ONE place, and the split rebuilt from the seed rather than inherited
>
> **check** · assert the two sizes exactly — if they differ, every comparison against the previous lecture is void
>
> ---
>
> **try** · restart the runtime and run this cell first, before anything else. A notebook that only runs because another one is still in memory is not reproducible, and restart-and-run-all is the only test of that.

In [ ]:
# Every import this notebook needs, in one place — a notebook that only runs
# because a previous one is still in memory is not reproducible.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import (GridSearchCV, KFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, add_dummy_feature
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt

income_cat = pd.cut(housing_full["median_income"],
                    bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5])
train_set, test_set = train_test_split(
    housing_full, test_size=0.2, random_state=RANDOM_STATE, stratify=income_cat)

housing  = train_set.copy()
X_train  = housing.drop(columns=["median_house_value"])
y_train  = housing["median_house_value"]
X_test   = test_set.drop(columns=["median_house_value"])
y_test   = test_set["median_house_value"]

assert len(X_train) == 16512 and len(X_test) == 4128
print("same split as the previous lecture — the seed guarantees it")

## 2 · What `LinearRegression().fit()` actually computed

Minimising $\lVert X\theta - y\rVert^2$ gives the normal equation

$$X^{\mathsf T}(X\hat\theta - y) = 0$$

Read row by row, that says the residual is orthogonal to **every column of X**.
Least squares is not an algebraic trick — it is a projection onto the column
space of $X$. Verify it rather than believing it:

> **Prompt · what LinearRegression().fit() actually computed**
>
> **input** · the numeric features with an intercept column added
>
> **output** · the normal-equation solution, and the residual's inner product with every column of X
>
> **constraint** · add the intercept column with `add_dummy_feature` BEFORE solving — without it the residual is not orthogonal to the constant and the assert fails for the wrong reason
>
> **check** · assert the largest |Xᵀ(Xθ̂ − y)| is negligible RELATIVE to the scale of y — an absolute tolerance on dollars is meaningless. Read the assert row by row: it says the residual is orthogonal to every column of X
>
> ---
>
> **try** · append a column that is the sum of two existing ones, and solve again. `np.linalg.inv` raises or returns nonsense; `np.linalg.pinv` does not. That is the invertibility condition, met in practice.

In [ ]:
num = X_train.select_dtypes(include=[np.number])
X = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).fit_transform(num)
X_b = add_dummy_feature(X)                     # the x0 = 1 column, for the intercept
y = y_train.values

theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
residual = X_b @ theta - y

# every column of X is orthogonal to the residual, to numerical precision
orth = X_b.T @ residual
print(f"largest |Xᵀ(Xθ̂ − y)| = {np.abs(orth).max():.3e}")
print(f"relative to the scale of y ({np.abs(y).mean():,.0f}): "
      f"{np.abs(orth).max() / np.abs(y).mean():.2e}")
assert np.abs(orth).max() / np.abs(y).mean() < 1e-6, "not orthogonal — check X"

`np.linalg.inv` is not what scikit-learn uses. It computes the pseudoinverse via
SVD, which still returns an answer when $X^{\mathsf T}X$ is singular — when you
have more features than instances, or when two columns are collinear. That is
the whole failure condition, and it is why you were warned against engineering a
feature as a weighted sum of existing ones.

## 3 · A number that means nothing

Fit an unconstrained decision tree, then score it on the very rows it was
fitted to. The result is not evidence that the tree is good — it is what any
model flexible enough to memorise will produce, and it is the reason the next
section exists.

> **Prompt · a training score on a model that can memorise**
>
> **input** · a ColumnTransformer, and an unconstrained decision tree
>
> **output** · the tree's RMSE on the rows it was fitted to
>
> **constraint** · score it on the TRAINING rows, deliberately, and say in the output what that means — this is a demonstration, not a result
>
> **check** · work out the number before running it: an unconstrained tree splits until every leaf is pure, so what can it score on the rows it split on?
>
> ---
>
> **try** · set `max_depth=4` and run it again. The training RMSE stops being zero — what has changed about the model, and has anything changed about the measurement?

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
preprocessing = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["ocean_proximity"]),
])

tree = Pipeline([("prep", preprocessing),
                 ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
tree.fit(X_train, y_train)
print(f"RMSE on the data it was fitted to: "
      f"${root_mean_squared_error(y_train, tree.predict(X_train)):,.0f}")
print("An unconstrained tree can put every training row in its own leaf.")

## 4 · Measure it honestly

`shuffle=True` is not decoration. The default `KFold` does **not** shuffle, so
two students whose dataframes are in different row orders get different folds
and cannot work out why their numbers disagree.

⏱ **about 90 seconds** — thirty fits in total, ten of them forests.

> **Prompt · ⏱ 90 s — measure it honestly**
>
> **input** · the three models, ten folds each
>
> **output** · mean, standard deviation and range of the fold RMSEs
>
> **constraint** · `shuffle=True` is NOT decoration — the default KFold does not shuffle, so two people whose dataframes are in different row orders get different folds and cannot work out why their numbers disagree
>
> **check** · print the fold minimum and maximum beside the mean. The spread decides which differences you are allowed to talk about: a mean of $50,000 from folds spanning $8,000 supports very different claims from one built from folds spanning $500
>
> ---
>
> **try** · drop `shuffle=True` and re-run. The mean moves. Which of the two numbers is right, and what does the question even mean?

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Linear regression": LinearRegression(),
    "Decision tree":     DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random forest":     RandomForestRegressor(n_estimators=100,
                                               random_state=RANDOM_STATE, n_jobs=-1),
}

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessing), ("model", model)])
    folds = -cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring="neg_root_mean_squared_error")
    results[name] = folds
    print(f"{name:20s} ${folds.mean():>9,.0f}  ± ${folds.std():>6,.0f}"
          f"   (folds ${folds.min():,.0f} – ${folds.max():,.0f})")

Report the spread, not just the mean. The folds span several thousand dollars,
so **any comparison that turns on less than a couple of thousand is not a
comparison.**

Compare the models on the *same* folds rather than comparing two averages —
paired differences remove the fold-to-fold variation that both models share:

> **Prompt · compare on the SAME folds**
>
> **input** · the two arrays of per-fold scores
>
> **output** · the paired difference, and how many folds each model wins
>
> **constraint** · subtract PER FOLD rather than comparing two averages — the folds differ in difficulty and both models feel that identically, so pairing cancels it
>
> **check** · the paired standard deviation should be much smaller than either model's own; and the win count, '10 of 10 folds', is an argument that a mean difference alone is not
>
> ---
>
> **try** · compare the two models the unpaired way instead — mean ± std against mean ± std. The intervals overlap and the comparison looks inconclusive. The paired version is not.

In [ ]:
diff = results["Random forest"] - results["Linear regression"]
# ddof=1: ten folds are a sample, and the slides quote the sample sd
print(f"forest − linear, per fold:  mean ${diff.mean():,.0f}  "
      f"sd ${diff.std(ddof=1):,.0f}")
print(f"folds where the forest wins: {(diff < 0).sum()}/10")

## 5 · Tune — on validation folds, never on the test set

⏱ **2–4 minutes.** Fifteen combinations × five folds = 75 forest fits. The
lecture's figure uses `cv=10`, which takes twice as long; five is enough here.

> **Prompt · ⏱ 3-6 min — tune on validation folds, never on the test set**
>
> **input** · fifteen combinations, ten folds each — 150 fits
>
> **output** · the best parameters and the best cross-validated RMSE
>
> **constraint** · the grid searches the WHOLE PIPELINE, so the preprocessing is refitted inside every fold — `model__` prefixes because the parameters belong to a step — and it reuses the SAME KFold object as the section above, so its number is comparable with the ones already printed
>
> **check** · detect whether the winner sits on the EDGE of the grid and say so — an optimum at the boundary means the optimum may lie outside it, and the search was too small
>
> ---
>
> **try** · read `best_score_` and ask what it is the score OF. It is the winner of a fifteen-way selection, measured on the folds that did the selecting, so it is optimistic by construction. The honest number is still three sections away.

In [ ]:
grid = {"model__max_features": [4, 6, 8, 10, 12],
        "model__n_estimators": [30, 100, 200]}

search = GridSearchCV(
    Pipeline([("prep", preprocessing),
              ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))]),
    grid, cv=cv,                     # the SAME ten folds as section 4, so the
                                     # numbers here are comparable with those
    scoring="neg_root_mean_squared_error", n_jobs=-1)
search.fit(X_train, y_train)

print(f"best {search.best_params_}")
print(f"best cross-validated RMSE ${-search.best_score_:,.0f}")

best_n = search.best_params_["model__n_estimators"]
if best_n == max(grid["model__n_estimators"]):
    print("\n⚠ the winner sits on the EDGE of the grid — the optimum may lie "
          "outside it. Search again with larger values.")

## 6 · What the encoder does with a category it has never seen

`ISLAND` — five districts in the whole state — is the category the first lecture
warned you about. With ten folds, some folds contain no ISLAND training row at
all, and `handle_unknown="ignore"` then encodes it as an all-zero column and
says nothing.

> **Prompt · what an unseen category actually encodes to**
>
> **input** · an encoder fitted without ISLAND, asked to transform ISLAND
>
> **output** · the resulting row, and its sum
>
> **constraint** · DEMONSTRATE it rather than describing it — fit without the category and transform with it
>
> **check** · every other district's encoded row sums to one. Work out what this one sums to, and what the model therefore sees
>
> ---
>
> **try** · set `handle_unknown='error'` instead and re-run. It raises — which is the same information, delivered loudly. Decide which you would rather have inside a cross-validation loop, and why the quiet option is still the right default here.

In [ ]:
enc = OneHotEncoder(handle_unknown="ignore").fit(
    housing[["ocean_proximity"]].query("ocean_proximity != 'ISLAND'"))
# a DataFrame, not a bare list: transform() warns about missing feature names
# otherwise, and a warning here would obscure the point of the cell
island = pd.DataFrame({"ocean_proximity": ["ISLAND"]})
row = enc.transform(island).toarray()[0]
print("an unseen category encodes to:", row)
print("sum:", row.sum(), "— and no warning, no error")

## 7 · The test set. Once.

Everything so far used only training data. This is the first and last time the
test set is touched.

> **Prompt · the test set, once**
>
> **input** · the 4,128 held-out districts
>
> **output** · the test RMSE with a 95% bootstrap interval, beside the cross-validated estimate
>
> **constraint** · bootstrap the SQUARED ERRORS and take the square root of the interval, rather than bootstrapping the RMSE directly — the mean is what the bootstrap is good at, and the square root comes afterwards. `method='percentile'` is not optional either: scipy defaults to BCa, and we are claiming the percentile bootstrap
>
> **check** · compare the test RMSE with the cross-validated estimate. Is the gap between them bigger or smaller than the interval — and what follows if it is smaller?
>
> ---
>
> **try** · nothing. This is the one cell in the course with no `try` line: the test set has now been used, and any change you make in response to its number is you fitting it.

In [ ]:
from scipy import stats

best = search.best_estimator_          # refitted on the whole training set
final_pred = best.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_pred)

squared = (final_pred - y_test.values) ** 2
# method= is not optional: scipy defaults to BCa, and we are claiming the
# PERCENTILE bootstrap. Name the estimator you mean.
lo, hi = np.sqrt(stats.bootstrap([squared], np.mean,
                                 confidence_level=0.95, method="percentile",
                                 random_state=RANDOM_STATE).confidence_interval)

print(f"test RMSE  ${final_rmse:,.0f}")
print(f"95% interval  ${lo:,.0f} – ${hi:,.0f}")
print(f"\ncross-validated estimate was ${-search.best_score_:,.0f}")
print("The two agree within the interval. A gap smaller than the interval is "
      "not evidence of anything.")

**Do not tune now.** If you adjust hyperparameters to improve that number you
are fitting the test set, and the improvement will not generalise. The number
you have is the number you report.

## 8 · One number for 4,128 districts is a summary, not a finding

Analysing the final model's errors is not tuning, and it is the part of the
report the client actually acts on. Break the error out by the categories they
care about.

> **Prompt · slice the error**
>
> **input** · the final model's test predictions
>
> **output** · RMSE and district count, broken down by ocean proximity and by income band
>
> **constraint** · report the COUNT beside every RMSE — a group of three districts and a group of 1,862 do not deserve the same weight in your conclusion — and pass `observed=True`, or pandas emits a row for every unobserved combination of categories
>
> **check** · one group is far worse than the rest, and its count is tiny. Before running it, predict which: the first lecture named a category with five districts in the whole state
>
> ---
>
> **try** · drop `observed=True`. Count the rows you get, and how many of them are NaN.

In [ ]:
err = pd.DataFrame({
    "error": final_pred - y_test.values,
    "ocean": test_set["ocean_proximity"].values,
    "income_cat": pd.cut(test_set["median_income"],
                         bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
                         labels=[1, 2, 3, 4, 5]).values,
})

def slice_by(col):
    g = err.groupby(col, observed=True)["error"]
    return pd.DataFrame({"n": g.size(),
                         "RMSE": g.apply(lambda e: np.sqrt((e ** 2).mean()))})

for col in ("income_cat", "ocean"):
    out = slice_by(col).sort_values("RMSE")
    out["RMSE"] = out["RMSE"].map(lambda v: f"${v:,.0f}")
    print(f"by {col}:"); print(out.to_string()); print()

Read the counts, not only the errors. The poorest band is predicted worst in
dollars — and its districts are the cheapest, so in *relative* terms it is worse
still. `ISLAND` has three districts in the test set, so its RMSE is an average
over three numbers and carries almost no information: it is a reminder that the
model was asked about a category it saw twice, not a measurement you could act
on.

**What you would tell the client:** the system is usable, and it should not be
deployed unqualified for the poorest districts, for the capped ones, or for
`ISLAND` at all.

## 9 · Where we are

- Fitting a linear model is orthogonal projection, and you verified the
  orthogonality numerically rather than taking it on trust.
- A score computed on the training rows cannot see overfitting. The tree proves
  it in one cell.
- Cross-validation gives an honest estimate **and** its spread, and two models
  differ only if the paired per-fold difference says so.
- All preprocessing lives inside the `Pipeline` handed to cross-validation, so
  leakage is structurally impossible rather than merely avoided.
- The test set was touched once, and the number came with an interval.

**Six questions to ask of any reported number** — yours or anyone else's:

1. Was every transformer fitted *after* the split, on training data only?
2. Is the preprocessing inside the pipeline passed to cross-validation?
3. Was the reported metric computed on held-out data?
4. Was the test set touched more than once?
5. Were hyperparameters selected using validation data, not test data?
6. Does any feature encode information unavailable at prediction time?

**Before the next lecture:** run this notebook top to bottom. Then change the
`KFold` from 10 splits to 3 and re-run from that cell. What happens to the mean
RMSE, and what happens to the spread — and which of the two changes should worry
you more?